### 🛠️ Environment Setup & Dependency Verification
This notebook includes verified dependencies to ensure reproducible execution.

- **Automatic Setup:** Cell 2 verifies Python version compatibility and installs verified package versions sequentially.
- **Network Notice:** Active internet access is required to download uncached packages.


In [1]:
# =====================================================================
# VERIFIED ENVIRONMENT DEPENDENCIES (2026-09-17 19:42:57)
# =====================================================================

import sys
import subprocess
import tempfile
import importlib.metadata

REQUIRED_PYTHON = (3, 12)
CURRENT_PYTHON = (sys.version_info.major, sys.version_info.minor)

# Major version mismatch -> Clean hard stop
if CURRENT_PYTHON[0] != REQUIRED_PYTHON[0]:
    req_major = REQUIRED_PYTHON[0]
    curr_major = CURRENT_PYTHON[0]
    print(f"❌ Error: Major Python version mismatch!")
    print(f"This notebook requires Python {req_major}.x, but your environment is running Python {curr_major}.x.\n")
    sys.exit("Execution stopped due to Python major version incompatibility.")

# Minor version mismatch -> Non-blocking warning
if CURRENT_PYTHON[1] != REQUIRED_PYTHON[1]:
    req_ver = f"{REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}"
    curr_ver = f"{CURRENT_PYTHON[0]}.{CURRENT_PYTHON[1]}"
    print(f"⚠️ This code was created with Python {req_ver}. You are trying to run it with {curr_ver}.")
    print(f"If installation fails, consider changing your runtime Python version back to {req_ver}.\n")

# Reproducibility manifest (dependencies, Python target, GPU context, integrity hash)
STEADY_PY_MANIFEST = {'python_version': {'major': 3, 'minor': 12}, 'dependencies': [{'name': 'humanize', 'version': '4.16.0', 'flags': []}], 'gpu': None, 'generated_at': '2026-09-17 19:42:57', 'tool_version': '44', 'dependency_hash': '2726570f0a936c9e6acb21298a3dacbe5166792d96a262113acafb611e236e68', 'raw_installs': []}

print(f"Applying verified environment dependencies [2026-09-17 19:42:57]...")
print("💡 Note: Dependencies are installed sequentially to prevent index conflicts.\n")

passed_count = 0
failed_packages = []
total_deps = len(STEADY_PY_MANIFEST["dependencies"])
installed_baseline = {}

def _run_pip_subprocess(cmd, timeout):
    captured = []
    returncode = 0
    try:
        with tempfile.TemporaryFile(mode="w+", encoding="utf-8", errors="replace") as tmp_out:
            proc = subprocess.run(cmd, stdin=subprocess.DEVNULL, stdout=tmp_out, stderr=subprocess.STDOUT, timeout=timeout)
            returncode = proc.returncode
            tmp_out.seek(0)
            for line in tmp_out.read().splitlines():
                if line.strip():
                    captured.append(line)
                    print(f"    {line}")
            sys.stdout.flush()
    except subprocess.TimeoutExpired:
        returncode = -1
        captured.append(f"Error: installation exceeded per-package timeout limit ({timeout}s).")
        print(f"    ❌ Installation timed out after {timeout}s.")
    except Exception as exc:
        returncode = -1
        captured.append(f"Execution failed: {exc}")
        print(f"    ❌ Execution failed: {exc}")
    return returncode, captured

for idx, item in enumerate(STEADY_PY_MANIFEST["dependencies"], start=1):
    name = item["name"]
    ver = item.get("version", "")
    flags = item.get("flags", [])
    specifier = f"{name}=={ver}" if ver else name

    # Step 1: Pre-install inspection
    # Avoids redundant re-installations in pre-configured platforms (Colab, Kaggle)
    already_satisfied = False
    try:
        current_ver = importlib.metadata.version(name)
        if not ver or current_ver == ver:
            already_satisfied = True
            passed_count += 1
            installed_baseline[name] = current_ver
            print(f"[{idx}/{total_deps}] ⚡ {name} ({current_ver}) already satisfied in environment")
    except Exception:
        pass

    if already_satisfied:
        continue

    # Step 2: Non-blocking installation via disk-backed stream redirection
    # Flags explanation:
    # - "--no-input": Prevents pip from prompting on stdin
    # - "--disable-pip-version-check": Eliminates overhead checking for newer pip releases
    # - "--no-warn-script-location": Suppresses path warnings for local bin paths
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-input",
        "--disable-pip-version-check",
        "--no-warn-script-location",
        specifier
    ] + flags

    print(f"[{idx}/{total_deps}] 📦 Installing {specifier}...")
    sys.stdout.flush()

    returncode, captured_output = _run_pip_subprocess(cmd, 120)

    if returncode == 0:
        passed_count += 1
        print(f"    ✅ {specifier} installed successfully")
        
        # Real-time drift audit across previously installed dependencies
        try:
            current_ver = importlib.metadata.version(name)
            installed_baseline[name] = current_ver
        except Exception:
            pass

        for prev_pkg, prev_ver in list(installed_baseline.items()):
            if prev_pkg == name:
                continue
            try:
                active_now = importlib.metadata.version(prev_pkg)
                if active_now != prev_ver:
                    print(f"   ⚠️ Dependency Drift: Installing '{specifier}' caused '{prev_pkg}' to drift from {prev_ver} ➔ {active_now}")
                    installed_baseline[prev_pkg] = active_now
            except Exception:
                pass
    else:
        err_snippet = captured_output[-1] if captured_output else "Unknown pip error"
        failed_packages.append((specifier, ver, flags, "\n".join(captured_output)))
        print(f"    ❌ {specifier} failed to install (exit code {returncode})")
        print(f"       ├─ Author Verified Version: {ver or 'unspecified'}")
        if flags:
            print(f"       ├─ Scoped Flags: {' '.join(flags)}")
        print(f"       └─ Error: {err_snippet}\n")

print("\n" + "=" * 60)
if not failed_packages:
    print(f"✅ Setup complete! All {passed_count}/{total_deps} dependencies verified.")
else:
    print(f"⚠️ Setup completed with issues: {passed_count}/{total_deps} packages installed.")
    print("Troubleshooting Steps:")
    print("1. Internet Access: Ensure your notebook environment has active internet access.")
    print("2. Unpinned Installs: Test installing failed libraries manually: '!pip install <pkg>'")
    print(f"3. Troubleshooting Steps: For a detailed guide on resolving setup errors, see: https://github.com/flyinacres/notebook_env/blob/main/HELP.md")

print("\n⚠️ Note: You may need to restart the kernel to use updated packages.")
print("=" * 60)


Applying verified environment dependencies [2026-09-17 19:42:57]...
💡 Note: Dependencies are installed sequentially to prevent index conflicts.

[1/1] 📦 Installing humanize==4.16.0...
       ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 2.1 MB/s eta 0:00:00
      Attempting uninstall: humanize
        Found existing installation: humanize 4.15.0
        Uninstalling humanize-4.15.0:
          Successfully uninstalled humanize-4.15.0
    ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
    bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
    ✅ humanize==4.16.0 installed successfully

✅ Setup complete! All 1/1 dependencies verified.

⚠️ Note: You may need to restart the kernel to use updated packages.


In [2]:
# Unmanaged pip cell: package already satisfied, no reinstall expected
%pip install humanize==4.16.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
import humanize
import importlib.metadata
print('ok', importlib.metadata.version('humanize'))

ok 4.16.0
